In [ ]:
#%run prelude.rc

import enum
import importlib.util
import sys
from pathlib import Path

import pyarrow


import time

import polars as pl
import numpy as np
import scipy.integrate as integrate
import HErmes as he
import HErmes.fitting as fit
import scipy.stats as st
import matplotlib

from scipy.spatial.transform import Rotation as rot
from datetime import datetime, UTC, timezone
from glob import glob

#pybindings
from pathlib import Path
import dashi as d
d.visual()
import tqdm


import matplotlib.pyplot as plt
import charmingbeauty as cb
lo = cb.layout
cb.visual.set_style_present()



import re
!export DJANGO_ALLOW_ASYNC_UNSAFE=1
import os
from matplotlib import font_manager
from matplotlib import rcParams


os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = '1'
plt.rcParams.update({'text.usetex' : False})


from matplotlib import font_manager


rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Open Sans']



import numpy as np
import polars as pl
import numpy as np
import polars as pl

def average_every_n_by_board(df, n, time_col="timestamp", board_col="board_id"):
    if len(df) == 0:
        return df

    if time_col not in df.columns:
        raise ValueError(f'{time_col} not in dataframe')
    if board_col not in df.columns:
        raise ValueError(f'{board_col} not in dataframe')

    out = []
    boards = np.unique(df[board_col].to_numpy())
       
    for b in boards:
        sub = (
            df.filter(pl.col(board_col) == b)
              .sort(time_col)
        )
        
        m = len(sub)
        if m == 0:
            continue

        # make consecutive bins AFTER sorting
        bin_id = np.arange(m) // n
        sub = sub.with_columns(pl.Series("bin_id", bin_id))
        
        exprs = []
        for c, dt in zip(sub.columns, sub.dtypes):
            if c in [board_col, "bin_id"]:
                continue
            if c == time_col:
                exprs.append(pl.col(c).mean().alias(c))
            elif dt.is_numeric():
                exprs.append(pl.col(c).mean().alias(c))
                
        agg = (
            sub.group_by("bin_id", maintain_order=True)
               .agg(exprs)
               .with_columns(pl.lit(b).alias(board_col))
               .drop("bin_id")
               .sort(time_col)
        )
        
        out.append(agg)

    if not out:
        return pl.DataFrame()

    return pl.concat(out).sort([board_col, time_col])





def average_every_n(df, n, time_col="timestamp"):
    if len(df) == 0:
        return df

    if time_col not in df.columns:
        raise ValueError(f'{time_col} not in dataframe')

    sub = df.sort(time_col)

    m = len(sub)
    bin_id = np.arange(m) // n
    sub = sub.with_columns(pl.Series("bin_id", bin_id))

    exprs = []
    for c, dt in zip(sub.columns, sub.dtypes):
        if c == "bin_id":
            continue
        if c == time_col:
            exprs.append(pl.col(c).mean().alias(c))
        elif dt.is_numeric():
            exprs.append(pl.col(c).mean().alias(c))

    return (
        sub.group_by("bin_id", maintain_order=True)
           .agg(exprs)
           .drop("bin_id")
           .sort(time_col)
    )

from pathlib import Path

def compress_df_by_timebin(df, dt=10.0, time_col="timestamp", board_col="board_id"):
    if len(df) == 0:
        return df

    df = df.sort([board_col, time_col])

    df = df.with_columns(
        (pl.col(time_col) / dt).floor().cast(pl.Int64).alias("tbin")
    )

    key_cols = [board_col, "tbin"]

    exprs = [
        pl.col(time_col).mean().alias(time_col)
    ]

    for c, dtp in zip(df.columns, df.dtypes):
        if c in key_cols or c == time_col:
            continue
        if dtp.is_numeric():
            exprs.append(pl.col(c).mean().alias(c))

    out = (
        df.group_by(key_cols)
          .agg(exprs)
          .drop("tbin")
          .sort([board_col, time_col])
    )

    return out




import polars as pl

def shift_local_time(df, last_t, dt, time_col="timestamp"):
    if len(df) == 0:
        return df, last_t

    first_local = float(df[time_col][0])
    offset = last_t + dt - first_local

    df = df.with_columns(
        (pl.col(time_col) + offset).alias(time_col)
    )

    new_last_t = float(df[time_col][-1])
    return df, new_last_t


import numpy as np
import polars as pl

def estimate_dt_by_board(df, board_col="board_id", time_col="timestamp", min_points=5):
    dt_map = {}

    if len(df) == 0:
        return dt_map

    boards = df[board_col].unique().to_list()

    for b in boards:
        sub = (
            df.filter(pl.col(board_col) == b)
              .sort(time_col)
        )

        t = sub[time_col].to_numpy()
        if len(t) < min_points:
            continue

        dt = np.diff(t)
        dt = dt[np.isfinite(dt) & (dt > 0)]

        if len(dt) == 0:
            continue

        dt_map[b] = float(np.median(dt))

    return dt_map

def estimate_dt(df, time_col="timestamp", min_points=5):
    if len(df) == 0:
        return None

    t = df.sort(time_col)[time_col].to_numpy()
    if len(t) < min_points:
        return None

    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]

    if len(dt) == 0:
        return None

    return float(np.median(dt))

def shift_local_time_by_board(df, last_t_map, dt_map, board_col="board_id", time_col="timestamp"):
    if len(df) == 0:
        return df, last_t_map

    out = []

    for b in df[board_col].unique().to_list():
        sub = (
            df.filter(pl.col(board_col) == b)
              .sort(time_col)
        )

        if len(sub) == 0:
            continue

        t0 = float(sub[time_col][0])

        last_t = last_t_map.get(b, None)
        dt = dt_map.get(b, None)

        if dt is None:
            # fallback: estimate from this chunk itself
            t = sub[time_col].to_numpy()
            d = np.diff(t)
            d = d[np.isfinite(d) & (d > 0)]
            dt = float(np.median(d)) if len(d) else 0.0

        if last_t is None:
            offset = -t0
        else:
            offset = last_t + dt - t0

        sub = sub.with_columns(
            (pl.col(time_col) + offset).alias(time_col)
        )

        last_t_map[b] = float(sub[time_col].max())
        out.append(sub)

    if not out:
        return df, last_t_map

    return pl.concat(out).sort([board_col, time_col]), last_t_map




def shift_local_time(df, last_t, dt, time_col="timestamp"):
    if len(df) == 0:
        return df, last_t

    df = df.sort(time_col)

    t0 = float(df[time_col][0])

    if dt is None:
        t = df[time_col].to_numpy()
        d = np.diff(t)
        d = d[np.isfinite(d) & (d > 0)]
        dt = float(np.median(d)) if len(d) else 0.0

    if last_t is None:
        offset = -t0
    else:
        offset = last_t + dt - t0

    df = df.with_columns(
        (pl.col(time_col) + offset).alias(time_col)
    )

    last_t = float(df[time_col].max())
    return df, last_t

import polars as pl
import gondola as gon
	
		
		

LTB_to_RB = {
    18: 3,
    2: 32,
    14: 31,
    23: 35,
    3: 23,
    25: 27,
    1: 19,
    4: 16,
    13: 8,
    15: 1,
    5: 26,
    22: 39,
    9: 9,
    7: 41,
    6: 2,
    12: 46,
    21: 7,
    20: 33,
    8: 36,
    11: 28,
}



# all are top rats besides 19; that is switched
PB_to_RB = {
    18: 3,
    2: 32,
    14: 31,
    23: 35,
    3: 23,
    25: 27,
    1: 19,
    4: 16,
    13: 8,
    15: 1,
    5: 26,
    22: 39,
    9: 9,
    7: 41,
    6: 2,
    12: 46,
    21: 7,
    20: 33,
    8: 36,
    11: 28,
}

RB_to_PB = {v: k for k, v in PB_to_RB.items()}

RB_to_LTB = {v: k for k, v in LTB_to_RB.items()}


RAT_to_RB = {
    1:  [3, 15],
    2:  [32, 14],
    3:  [31, 29],
    4:  [35, 13],
    5:  [23, 21],
    6:  [27, 24],
    7:  [20, 19],
    8:  [16, 25],
    9:  [8, 30],
    10: [1, 11],
    11: [26, 22],
    12: [39, 40],
    13: [9, 18],
    14: [41, 42],
    15: [2, 4],
    16: [46, 44],
    17: [7, 17],
    18: [33, 34],
    19: [36, 6],
    20: [28, 5],
}


RB_to_RAT = {}

for rb, rats in RAT_to_RB.items():
    for rat in rats:
        RB_to_RAT[rat] = rb
        

In [ ]:
def build_paddle_map():
    raw = """
1	A   04-11	16
1	B	04-12	16
2	A	04-09	16
2	B	04-10	16
3	A	04-07	16
3	B	04-08	16
4	A	04-05	16
4	B	04-06	16
5	A	04-03	16
5	B	04-04	16
6	A	04-01	16
6	B	04-02	16
7	A	12-16	46
7	B	12-15	46
8	A	12-14	46
8	B	12-13	46
9	A	12-12	46
9	B	12-11	46
10	A	12-10	46
10	B	12-09	46
11	A	12-08	46
11	B	12-07	46
12	A	12-06	46
12	B	12-05	46
13	A	15-02	1
13	B	15-01	1
14	A	15-04	1
14	B	15-03	1
15	A	15-06	1
15	B	15-05	1
16	A	15-08	1
16	B	15-07	1
17	A	15-10	1
17	B	15-09	1
18	A	15-12	1
18	B	15-11	1
19	A	07-05	41
19	B	07-06	41
20	A	07-07	41
20	B	07-08	41
21	A	07-09	41
21	B	07-10	41
22	A	07-11	41
22	B	07-12	41
23	A	07-13	41
23	B	07-14	41
24	A	07-15	41
24	B	07-16	41
25	A	04-14	16
25	B	04-13	16
26	A	04-16	16
26	B	04-15	16
27	A	13-12	8
27	B	13-11	8
28	A	13-10	8
28	B	13-09	8
29	A	13-08	8
29	B	13-07	8
30	A	13-06	8
30	B	13-05	8
31	A	13-04	8
31	B	13-03	8
32	A	13-02	8
32	B	13-01	8
33	A	22-10	39
33	B	22-09	39
34	A	22-08	39
34	B	22-07	39
35	A	22-12	39
35	B	22-11	39
36	A	22-06	39
36	B	22-05	39
37	A	22-14	39
37	B	22-13	39
38	A	22-04	39
38	B	22-03	39
39	A	22-16	39
39	B	22-15	39
40	A	22-02	39
40	B	22-01	39
41	A	12-04	46
41	B	12-03	46
42	A	12-02	46
42	B	12-01	46
43	A	06-06	2
43	B	06-05	2
44	A	06-08	2
44	B	06-07	2
45	A	06-10	2
45	B	06-09	2
46	A	06-12	2
46	B	06-11	2
47	A	06-14	2
47	B	06-13	2
48	A	06-16	2
48	B	06-15	2
49	A	08-10	36
49	B	08-09	36
50	A	08-08	36
50	B	08-07	36
51	A	08-12	36
51	B	08-11	36
52	A	08-06	36
52	B	08-05	36
53	A	08-14	36
53	B	08-13	36
54	A	08-04	36
54	B	08-03	36
55	A	08-16	36
55	B	08-15	36
56	A	08-02	36
56	B	08-01	36
57	A	05-04	26
57	B	05-03	26
58	A	09-14	9
58	B	09-13	9
59	A	21-04	7
59	B	21-03	7
60	A	19-14	20
60	B	19-13	20
61	A	18-11	3
61	B	18-12	3
62	A	18-09	3
62	B	18-10	3
63	A	18-07	3
63	B	18-08	3
64	A	18-05	3
64	B	18-06	3
65	A	18-03	3
65	B	18-04	3
66	A	18-01	3
66	B	18-02	3
67	A	02-02	32
67	B	02-01	32
68	A	02-04	32
68	B	02-03	32
69	A	02-06	32
69	B	02-05	32
70	A	02-08	32
70	B	02-07	32
71	A	02-10	32
71	B	02-09	32
72	A	02-12	32
72	B	02-11	32
73	A	18-13	3
73	B	18-14	3
74	A	18-15	3
74	B	18-16	3
75	A	14-01	31
75	B	14-02	31
76	A	14-03	31
76	B	14-04	31
77	A	14-05	31
77	B	14-06	31
78	A	14-07	31
78	B	14-08	31
79	A	03-15	23
79	B	03-16	23
80	A	03-13	23
80	B	03-14	23
81	A	03-11	23
81	B	03-12	23
82	A	03-09	23
82	B	03-10	23
83	A	03-07	23
83	B	03-08	23
84	A	03-05	23
84	B	03-06	23
85	A	03-03	23
85	B	03-04	23
86	A	03-01	23
86	B	03-02	23
87	A	23-15	35
87	B	23-16	35
88	A	23-13	35
88	B	23-14	35
89	A	23-11	35
89	B	23-12	35
90	A	23-09	35
90	B	23-10	35
91	A	02-14	32
91	B	02-13	32
92	A	02-16	32
92	B	02-15	32
93	A	23-01	35
93	B	23-02	35
94	A	23-03	35
94	B	23-04	35
95	A	23-05	35
95	B	23-06	35
96	A	23-07	35
96	B	23-08	35
97	A	25-15	27
97	B	25-16	27
98	A	25-13	27
98	B	25-14	27
99	A	25-11	27
99	B	25-12	27
100	A	25-09	27
100	B	25-10	27
101	A	25-07	27
101	B	25-08	27
102	A	25-05	27
102	B	25-06	27
103	A	25-03	27
103	B	25-04	27
104	A	25-01	27
104	B	25-02	27
105	A	14-15	31
105	B	14-16	31
106	A	14-13	31
106	B	14-14	31
107	A	14-11	31
107	B	14-12	31
108	A	14-09	31
108	B	14-10	31
109	A	13-16	8
109	B	13-15	8
110	A	13-14	8
110	B	13-13	8
111	A	15-16	1
111	B	15-15	1
112	A	15-14	1
112	B	15-13	1
113	A	05-10	26
113	B	05-09	26
114	A	05-08	26
114	B	05-07	26
115	A	05-06	26
115	B	05-05	26
116	A	19-02	20
116	B	19-01	20
117	A	19-04	20
117	B	19-03	20
118	A	19-06	20
118	B	19-05	20
119	A	11-16	28
119	B	11-15	28
120	A	11-14	28
120	B	11-13	28
121	A	11-12	28
121	B	11-11	28
122	A	11-10	28
122	B	11-09	28
123	A	11-08	28
123	B	11-07	28
124	A	11-06	28
124	B	11-05	28
125	A	11-04	28
125	B	11-03	28
126	A	11-02	28
126	B	11-01	28
127	A	09-16	9
127	B	09-15	9
128	A	05-02	26
128	B	05-01	26
129	A	06-02	2
129	B	06-01	2
130	A	06-04	2
130	B	06-03	2
131	A	07-02	41
131	B	07-01	41
132	A	07-04	41
132	B	07-03	41
133	A	09-08	9
133	B	09-07	9
134	A	09-10	9
134	B	09-09	9
135	A	09-12	9
135	B	09-11	9
136	A	21-16	7
136	B	21-15	7
137	A	21-14	7
137	B	21-13	7
138	A	21-12	7
138	B	21-11	7
139	A	20-02	33
139	B	20-01	33
140	A	20-04	33
140	B	20-03	33
141	A	20-06	33
141	B	20-05	33
142	A	20-08	33
142	B	20-07	33
143	A	20-10	33
143	B	20-09	33
144	A	20-12	33
144	B	20-11	33
145	A	20-14	33
145	B	20-13	33
146	A	20-16	33
146	B	20-15	33
147	A	21-02	7
147	B	21-01	7
148	A	19-16	20
148	B	19-15	20
149	A	05-12	26
149	B	05-11	26
150	A	05-14	26
150	B	05-13	26
151	A	05-16	26
151	B	05-15	26
152	A	09-01	9
152	B	09-02	9
153	A	09-03	9
153	B	09-04	9
154	A	09-05	9
154	B	09-06	9
155	A	21-05	7
155	B	21-06	7
156	A	21-07	7
156	B	21-08	7
157	A	21-09	7
157	B	21-10	7
158	A	19-08	20
158	B	19-07	20
159	A	19-10	20
159	B	19-09	20
160	A	19-12	20
160	B	19-11	20
""".strip().splitlines()

    paddle_map = {}

    for line in raw:
        parts = line.split()
        paddle = int(parts[0])
        side   = parts[1]
        pb_ch  = parts[2]
        rb     = int(parts[3])

        pb, ch = pb_ch.split("-")
        pb = int(pb)
        ch = int(ch)

        signed_id = -paddle if side == "A" else paddle

        paddle_map[signed_id] = {
            "rb": rb,
            "pb": pb,
            "ch": ch
        }

    return paddle_map


paddle_map = build_paddle_map()

def build_pbch_to_paddle_map(paddle_map):
    pbch_to_paddle = {}

    for signed_pid, info in paddle_map.items():
        key = (info["pb"], info["ch"])

        if key in pbch_to_paddle:
            raise ValueError(f"Duplicate mapping for {key}")

        pbch_to_paddle[key] = signed_pid

    return pbch_to_paddle


pbch_to_paddle = build_pbch_to_paddle_map(paddle_map)

# -------------------------------------------------
# paddle_id (signed) -> (ltb_id, ltb_channel)
# A side = negative paddle_id
# B side = positive paddle_id
# -------------------------------------------------

paddle_to_ltb = {}

def add(pid, side, ltb, ch):
    key = -pid if side == "A" else pid
    paddle_to_ltb[key] = (ltb, ch)


# --- fill map ---
add(1,"A",8,11); add(1,"B",8,12)
add(2,"A",8,9);  add(2,"B",8,10)
add(3,"A",8,7);  add(3,"B",8,8)
add(4,"A",8,5);  add(4,"B",8,6)
add(5,"A",8,3);  add(5,"B",8,4)
add(6,"A",8,1);  add(6,"B",8,2)
add(7,"A",16,16); add(7,"B",16,15)
add(8,"A",16,14); add(8,"B",16,13)
add(9,"A",16,12); add(9,"B",16,11)
add(10,"A",16,10); add(10,"B",16,9)
add(11,"A",16,8); add(11,"B",16,7)
add(12,"A",16,6); add(12,"B",16,5)

add(13,"A",10,2); add(13,"B",10,1)
add(14,"A",10,4); add(14,"B",10,3)
add(15,"A",10,6); add(15,"B",10,5)
add(16,"A",10,8); add(16,"B",10,7)
add(17,"A",10,10); add(17,"B",10,9)
add(18,"A",10,12); add(18,"B",10,11)

add(19,"A",14,5); add(19,"B",14,6)
add(20,"A",14,7); add(20,"B",14,8)
add(21,"A",14,9); add(21,"B",14,10)
add(22,"A",14,11); add(22,"B",14,12)
add(23,"A",14,13); add(23,"B",14,14)
add(24,"A",14,15); add(24,"B",14,16)

add(25,"A",8,14); add(25,"B",8,13)
add(26,"A",8,16); add(26,"B",8,15)

add(27,"A",9,12); add(27,"B",9,11)
add(28,"A",9,10); add(28,"B",9,9)
add(29,"A",9,8); add(29,"B",9,7)
add(30,"A",9,6); add(30,"B",9,5)
add(31,"A",9,4); add(31,"B",9,3)
add(32,"A",9,2); add(32,"B",9,1)

add(33,"A",12,10); add(33,"B",12,9)
add(34,"A",12,8); add(34,"B",12,7)
add(35,"A",12,12); add(35,"B",12,11)
add(36,"A",12,6); add(36,"B",12,5)
add(37,"A",12,14); add(37,"B",12,13)
add(38,"A",12,4); add(38,"B",12,3)
add(39,"A",12,16); add(39,"B",12,15)
add(40,"A",12,2); add(40,"B",12,1)

add(41,"A",16,4); add(41,"B",16,3)
add(42,"A",16,2); add(42,"B",16,1)

add(43,"A",15,6); add(43,"B",15,5)
add(44,"A",15,8); add(44,"B",15,7)
add(45,"A",15,10); add(45,"B",15,9)
add(46,"A",15,12); add(46,"B",15,11)
add(47,"A",15,14); add(47,"B",15,13)
add(48,"A",15,16); add(48,"B",15,15)

add(49,"A",19,10); add(49,"B",19,9)
add(50,"A",19,8); add(50,"B",19,7)
add(51,"A",19,12); add(51,"B",19,11)
add(52,"A",19,6); add(52,"B",19,5)
add(53,"A",19,14); add(53,"B",19,13)
add(54,"A",19,4); add(54,"B",19,3)
add(55,"A",19,16); add(55,"B",19,15)
add(56,"A",19,2); add(56,"B",19,1)

add(57,"A",11,4); add(57,"B",11,3)
add(58,"A",13,14); add(58,"B",13,13)
add(59,"A",17,4); add(59,"B",17,3)
add(60,"A",7,14); add(60,"B",7,13)

add(61,"A",1,11); add(61,"B",1,12)
add(62,"A",1,9); add(62,"B",1,10)
add(63,"A",1,7); add(63,"B",1,8)
add(64,"A",1,5); add(64,"B",1,6)
add(65,"A",1,3); add(65,"B",1,4)
add(66,"A",1,1); add(66,"B",1,2)

add(67,"A",2,2); add(67,"B",2,1)
add(68,"A",2,4); add(68,"B",2,3)
add(69,"A",2,6); add(69,"B",2,5)
add(70,"A",2,8); add(70,"B",2,7)
add(71,"A",2,10); add(71,"B",2,9)
add(72,"A",2,12); add(72,"B",2,11)

add(73,"A",1,13); add(73,"B",1,14)
add(74,"A",1,15); add(74,"B",1,16)

add(75,"A",3,1); add(75,"B",3,2)
add(76,"A",3,3); add(76,"B",3,4)
add(77,"A",3,5); add(77,"B",3,6)
add(78,"A",3,7); add(78,"B",3,8)

add(79,"A",5,15); add(79,"B",5,16)
add(80,"A",5,13); add(80,"B",5,14)
add(81,"A",5,11); add(81,"B",5,12)
add(82,"A",5,9); add(82,"B",5,10)
add(83,"A",5,7); add(83,"B",5,8)
add(84,"A",5,5); add(84,"B",5,6)
add(85,"A",5,3); add(85,"B",5,4)
add(86,"A",5,1); add(86,"B",5,2)

add(87,"A",4,15); add(87,"B",4,16)
add(88,"A",4,13); add(88,"B",4,14)
add(89,"A",4,11); add(89,"B",4,12)
add(90,"A",4,9); add(90,"B",4,10)

add(91,"A",2,14); add(91,"B",2,13)
add(92,"A",2,16); add(92,"B",2,15)

add(93,"A",4,1); add(93,"B",4,2)
add(94,"A",4,3); add(94,"B",4,4)
add(95,"A",4,5); add(95,"B",4,6)
add(96,"A",4,7); add(96,"B",4,8)

add(97,"A",6,15); add(97,"B",6,16)
add(98,"A",6,13); add(98,"B",6,14)
add(99,"A",6,11); add(99,"B",6,12)
add(100,"A",6,9); add(100,"B",6,10)
add(101,"A",6,7); add(101,"B",6,8)
add(102,"A",6,5); add(102,"B",6,6)
add(103,"A",6,3); add(103,"B",6,4)
add(104,"A",6,1); add(104,"B",6,2)

add(105,"A",3,15); add(105,"B",3,16)
add(106,"A",3,13); add(106,"B",3,14)
add(107,"A",3,11); add(107,"B",3,12)
add(108,"A",3,9); add(108,"B",3,10)

add(109,"A",9,16); add(109,"B",9,15)
add(110,"A",9,14); add(110,"B",9,13)

add(111,"A",10,16); add(111,"B",10,15)
add(112,"A",10,14); add(112,"B",10,13)

add(113,"A",11,10); add(113,"B",11,9)
add(114,"A",11,8); add(114,"B",11,7)
add(115,"A",11,6); add(115,"B",11,5)

add(116,"A",7,2); add(116,"B",7,1)
add(117,"A",7,4); add(117,"B",7,3)
add(118,"A",7,6); add(118,"B",7,5)

add(119,"A",20,16); add(119,"B",20,15)
add(120,"A",20,14); add(120,"B",20,13)
add(121,"A",20,12); add(121,"B",20,11)
add(122,"A",20,10); add(122,"B",20,9)
add(123,"A",20,8); add(123,"B",20,7)
add(124,"A",20,6); add(124,"B",20,5)
add(125,"A",20,4); add(125,"B",20,3)
add(126,"A",20,2); add(126,"B",20,1)

add(127,"A",13,16); add(127,"B",13,15)
add(128,"A",11,2); add(128,"B",11,1)

add(129,"A",15,2); add(129,"B",15,1)
add(130,"A",15,4); add(130,"B",15,3)

add(131,"A",14,2); add(131,"B",14,1)
add(132,"A",14,4); add(132,"B",14,3)

add(133,"A",13,8); add(133,"B",13,7)
add(134,"A",13,10); add(134,"B",13,9)
add(135,"A",13,12); add(135,"B",13,11)

add(136,"A",17,16); add(136,"B",17,15)
add(137,"A",17,14); add(137,"B",17,13)
add(138,"A",17,12); add(138,"B",17,11)

add(139,"A",18,2); add(139,"B",18,1)
add(140,"A",18,4); add(140,"B",18,3)
add(141,"A",18,6); add(141,"B",18,5)
add(142,"A",18,8); add(142,"B",18,7)
add(143,"A",18,10); add(143,"B",18,9)
add(144,"A",18,12); add(144,"B",18,11)
add(145,"A",18,14); add(145,"B",18,13)
add(146,"A",18,16); add(146,"B",18,15)

add(147,"A",17,2); add(147,"B",17,1)

add(148,"A",7,16); add(148,"B",7,15)

add(149,"A",11,12); add(149,"B",11,11)
add(150,"A",11,14); add(150,"B",11,13)
add(151,"A",11,16); add(151,"B",11,15)

add(152,"A",13,1); add(152,"B",13,2)
add(153,"A",13,3); add(153,"B",13,4)
add(154,"A",13,5); add(154,"B",13,6)

add(155,"A",17,5); add(155,"B",17,6)
add(156,"A",17,7); add(156,"B",17,8)
add(157,"A",17,9); add(157,"B",17,10)

add(158,"A",7,8); add(158,"B",7,7)
add(159,"A",7,10); add(159,"B",7,9)
add(160,"A",7,12); add(160,"B",7,11)

ltb_to_paddles = {}

for pid_signed, (ltb, ch) in paddle_to_ltb.items():
    key = (ltb, ch)
    if key not in ltb_to_paddles:
        ltb_to_paddles[key] = 0

    ltb_to_paddles[key] = pid_signed

In [ ]:
print("gondola version:", gon.__version__)
print("gondola path:", gon.__file__)

db = gon.db


paddles = db.TofPaddle.all()
#print(paddles[1].__dir__())

#(DSI,J, channel) -> (paddle_ID, panel_ID)

dsiJ_chP = db.get_dsi_j_ch_pid_map()
'''for key in dsiJ_chP.keys():
    for key2 in dsiJ_chP[key]:
        for key2 in dsiJ_chP[key]:
            print(key,key2, dsiJ_chP[key][key2])
            print("\n\n")
'''            
pid_dsiJch = {}

for dsi in dsiJ_chP:
    for j in dsiJ_chP[dsi]:
        for ch, (pid, panel) in dsiJ_chP[dsi][j].items():
            key = (pid, panel)
            val = (dsi, j, ch)
            if key not in pid_dsiJch:
                pid_dsiJch[key] = []
            pid_dsiJch[key].append(val)



from collections import defaultdict

# paddles from DB
paddles = db.TofPaddle.all()





# -------------------------------------------------
# (DSI, J) -> list of RB IDs
# because each DSI/J line can have 2 RBs
# -------------------------------------------------
Paddle_to_RAT = defaultdict(set)

dsiJ_to_RBs = defaultdict(set)

for p in paddles:
    key = (int(p.dsi), int(p.j_rb))
    rb_id = int(p.rb_id)
    
    dsiJ_to_RBs[key].add(rb_id)
    Paddle_to_RAT[p.paddle_id].add(RB_to_RAT[p.rb_id])
    

# convert sets to sorted lists
dsiJ_to_RBs = {
    key: sorted(list(rbs))
    for key, rbs in dsiJ_to_RBs.items()
}

print("\n=== (DSI, J) -> RBs ===")
for key in sorted(dsiJ_to_RBs):
    print(f"{key} -> {dsiJ_to_RBs[key]}")


# -------------------------------------------------
# inverse: RB ID -> list of (DSI, J)
# -------------------------------------------------
RB_to_dsiJ = defaultdict(set)

for dsiJ, rbs in dsiJ_to_RBs.items():
    for rb_id in rbs:
        RB_to_dsiJ[rb_id].add(dsiJ)

RB_to_dsiJ = {
    rb_id: sorted(list(dsiJs))
    for rb_id, dsiJs in RB_to_dsiJ.items()
}

In [ ]:
%matplotlib inline


plt.rcParams["font.family"] = "DejaVu Sans"
import gondola as gon
import time

import channel_rates

starttime = 1766959800 #1766115801 # <- this is 10066 #test 1766959800
endtime =   1766969800#1767979800 # <- end   #test 1766979800
'''files = gon.io.grace_get_telemetry_binaries(
    starttime, #start 1765835400
    endtime, #1766949800     1767039800 like 22 hours...
    #, #end time 1767979800 (testing it is for the random 100,000 seconds of flight)
    '/home/gaps/tof-data/antarctica/nextcloud/flight_2025-26'
)'''


chunk_size = 500

# global accumulated outputs
dfPB = dfPA = dfCPU = dfRB = dfLTB = dfMTB = None

# running last times
pa_last_t_map = {}
rb_last_t_map = {}
ltb_last_t_map = {}

cpu_last_t = None
mtb_last_t = None

# cadence maps
pa_dt_map = {}
rb_dt_map = {}
ltb_dt_map = {}

cpu_dt = None
mtb_dt = None


lpt = None
toml_find = False


dfSIP = dfPA = dfCPU = dfRB = dfLTB = dfMTB = None

pa_last_t  = -2.0
cpu_last_t = -2.0
rb_last_t  = -10.0
ltb_last_t = -2.0
mtb_last_t = -10.0
sip_last_t = -30
pb_last_t = -2.0


pa_dt  = 2.0
cpu_dt = 2.0
rb_dt  = 10.0
ltb_dt = 2.0
mtb_dt = 10.0
sip_dt = 30
pb_dt = 2






In [ ]:

#incase we dont want to run again and just load vals in
pa_dt_map = {1.0: 20.0, 2.0: 20.0, 3.0: 20.0, 7.0: 20.0, 9.0: 20.0, 16.0: 20.0, 19.0: 20.0, 23.0: 20.0, 26.0: 20.0, 28.0: 20.0, 31.0: 20.0, 32.0: 20.0, 33.0: 20.0, 35.0: 20.0, 39.0: 20.0, 41.0: 20.0, 46.0: 20.0}
rb_dt_map = {1.0: 10.0, 2.0: 10.0, 3.0: 10.0, 4.0: 10.0, 5.0: 10.0, 6.0: 10.0, 7.0: 10.0, 8.0: 10.0, 9.0: 10.0, 11.0: 10.0, 14.0: 10.0, 15.0: 10.0, 16.0: 10.0, 17.0: 10.0, 18.0: 10.0, 19.0: 10.0, 20.0: 10.0, 21.0: 10.0, 22.0: 10.0, 23.0: 10.0, 24.0: 10.0, 25.0: 10.0, 26.0: 10.0, 28.0: 10.0, 30.0: 10.0, 31.0: 10.0, 32.0: 10.0, 33.0: 10.0, 34.0: 10.0, 35.0: 10.0, 39.0: 10.0, 40.0: 10.0, 41.0: 10.0, 42.0: 10.0, 44.0: 10.0, 46.0: 10.0}
ltb_dt_map = {1.0: 20.0, 2.0: 20.0, 3.0: 20.0, 7.0: 20.0, 8.0: 20.0, 9.0: 20.0, 16.0: 20.0, 19.0: 20.0, 23.0: 20.0, 26.0: 20.0, 28.0: 20.0, 31.0: 20.0, 32.0: 20.0, 33.0: 20.0, 35.0: 20.0, 39.0: 20.0, 41.0: 20.0, 46.0: 20.0}
cpu_dt = 2.0
mtb_dt = 10.0


print("PA dt map:", pa_dt_map)
print("RB dt map:", rb_dt_map)
print("LTB dt map:", ltb_dt_map)
print("CPU dt:", cpu_dt)
print("MTB dt:", mtb_dt)


#sipM.timestamps looks like 30 seconds

plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # always available



dfPA  = pl.read_parquet("saved_dfs_absolute_ts/dfPA.parquet")
dfPB  = pl.read_parquet("saved_dfs_absolute_ts/dfPB.parquet")
dfRB  = pl.read_parquet("saved_dfs_absolute_ts/dfRB.parquet")
dfLTB = pl.read_parquet("saved_dfs_absolute_ts/dfLTB.parquet")
dfMTB = pl.read_parquet("saved_dfs_absolute_ts/dfMTB.parquet")
dfCPU = pl.read_parquet("saved_dfs_absolute_ts/dfCPU.parquet")
dfSIP = pl.read_parquet("saved_dfs_absolute_ts/dfSIP.parquet")
dfRates = pl.read_parquet("saved_dfs_absolute_ts/paddle_rates.parquet")

paddleRateTS = dfRates["timestamp"].to_numpy().astype(np.int64)


paddleRates = {
    int(c.replace("paddle_", "")): dfRates[c].to_numpy()
    for c in dfRates.columns if c.startswith("paddle_")
}




dfRatesHit  = pl.read_parquet("saved_dfs_absolute_ts/paddle_rates_hit.parquet")
dfRatesBeta = pl.read_parquet("saved_dfs_absolute_ts/paddle_rates_beta.parquet")

paddleRateTS = dfRatesHit["timestamp"].to_numpy()

paddleRatesHit = {
    int(c.replace("paddle_", "")): dfRatesHit[c].to_numpy()
    for c in dfRatesHit.columns if c.startswith("paddle_")
}

paddleRatesBeta = {
    int(c.replace("paddle_", "")): dfRatesBeta[c].to_numpy()
    for c in dfRatesBeta.columns if c.startswith("paddle_")
}




In [ ]:
pa_temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
pa_bias_cols = [c for c in dfPA.columns if c.startswith("biases")]

rb_temp_cols = [c for c in dfRB.columns if c.startswith("tmp_")]
rb_voltage_cols = [c for c in dfRB.columns if c.endswith("_voltage")]
rb_current_cols = [c for c in dfRB.columns if c.endswith("_current")]
rb_power_cols = [c for c in dfRB.columns if c.endswith("_power")]
rb_env_cols = ["pressure", "humidity"]

cpu_temp_cols = [c for c in dfCPU.columns if "temp" in c.lower()]
cpu_freq_cols = [c for c in dfCPU.columns if "freq" in c.lower()]

ltb_reasonable_cols = ["trenz_temp", "ltb_temp", "thresh0", "thresh1", "thresh2"]

mtb_rate_cols = [
    "trate", "lost_trate", "rb_lost_rate", "tiu_busy_rate",
    "trg_lost_trg_rate", "gaps_blocked_rate", "track_blocked_rate",
    "any_blocked_rate", "trkctrl_blocked_rate"
]



def finite_mask(*arrays):
    mask = np.ones(len(arrays[0]), dtype=bool)
    for a in arrays:
        a = np.asarray(a)
        mask &= np.isfinite(a)
    return mask

def range_mask(x, xmin=None, xmax=None):
    x = np.asarray(x)
    mask = np.isfinite(x)
    if xmin is not None:
        mask &= x >= xmin
    if xmax is not None:
        mask &= x <= xmax
    return mask


# PA
PA_TEMP_MIN, PA_TEMP_MAX = -50, 60
PA_BIAS_MIN, PA_BIAS_MAX = 45, 65

# RB
RB_TEMP_MIN, RB_TEMP_MAX = -50, 90
RB_VOLT_MIN, RB_VOLT_MAX = -5, 10
RB_CURR_MIN, RB_CURR_MAX = -1, 10
RB_PWR_MIN,  RB_PWR_MAX  = -1, 50
HUM_MIN, HUM_MAX = 0, 100
PRESS_MIN, PRESS_MAX = 0, 1200

# CPU
CPU_TEMP_MIN, CPU_TEMP_MAX = -20, 120
CPU_FREQ_MIN, CPU_FREQ_MAX = 0, 5000

# LTB
LTB_TEMP_MIN, LTB_TEMP_MAX = -50, 90
THR_MIN, THR_MAX = 0, 1000

# MTB
RATE_MIN, RATE_MAX = 0, 1e6





In [ ]:


'''

import os


outdir = "saved_dfs"
os.makedirs(outdir, exist_ok=True)

dfPA.write_parquet(f"{outdir}/dfPA.parquet")
dfCPU.write_parquet(f"{outdir}/dfCPU.parquet")
dfRB.write_parquet(f"{outdir}/dfRB.parquet")
dfLTB.write_parquet(f"{outdir}/dfLTB.parquet")
dfMTB.write_parquet(f"{outdir}/dfMTB.parquet")
dfSIP.write_parquet(f"{outdir}/dfSIP.parquet")
dfPB.write_parquet(f"{outdir}/dfPB.parquet")
#takes reallly long to go through everything...
'''



In [ ]:
# helper
%matplotlib inline

def get_time(df):
    if "timestamp" in df.columns:
        return df["timestamp"].to_numpy()
    if "total_elapsed" in df.columns:
        return df["total_elapsed"].to_numpy()
    raise KeyError(f"No timestamp-like column found. Columns: {df.columns}")



In [ ]:
def corr_subset(df, cols, cuts=None):
    arrs = []
    names = []
    n = len(df)
    mask = np.ones(n, dtype=bool)
    
    for c in cols:
        x = df[c].to_numpy()
        mask &= np.isfinite(x)
        if cuts and c in cuts:
            lo, hi = cuts[c]
            if lo is not None:
                mask &= x >= lo
            if hi is not None:
                mask &= x <= hi
    
    for c in cols:
        arrs.append(df[c].to_numpy()[mask])
        names.append(c)
        
    A = np.column_stack(arrs)
    C = np.corrcoef(A, rowvar=False)
    return names, C
    
def plot_corr(ax, names, C, title):
    im = ax.imshow(C, vmin=-1, vmax=1)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=8)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_title(title)
    return im

In [ ]:
plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # always available




def nearest_match(x_ref, x_other, max_dt=30):
    x_ref = np.asarray(x_ref)
    x_other = np.asarray(x_other)
    
    if len(x_other) == 0:
        raise ValueError("nearest_match: reference comparison array is empty")
        
    if len(x_other) == 1:
        idx = np.zeros(len(x_ref), dtype=int)
        dt = np.abs(x_other[0] - x_ref)
    else:
        idx = np.searchsorted(x_other, x_ref)
        idx = np.clip(idx, 1, len(x_other) - 1)

        left = idx - 1
        right = idx

        choose_right = np.abs(x_other[right] - x_ref) < np.abs(x_other[left] - x_ref)
        idx = np.where(choose_right, right, left)

        dt = np.abs(x_other[idx] - x_ref)

    # apply max time cut
    if max_dt is not None:
        idx = np.where(dt <= max_dt, idx, -1)

    return idx, dt


def getTempAv(dfPA):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
    
    # stack into matrix: shape (n_rows, 16)
    temps = np.column_stack([dfPA[c].to_numpy() for c in temp_cols])
    # ignore NaNs automatically
    return np.nanmean(temps, axis=1)

    
def getTempAvB(dfPA, boardID):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
    
    sub = (
        dfPA
        .filter(pl.col("board_id") == boardID)
        .group_by("timestamp")
        .agg([
            *[pl.col(c).mean().alias(c) for c in temp_cols]
        ])
        .sort("timestamp")
    )
    
    if len(sub) == 0:
        return np.array([]), np.array([])
    
    temps = np.column_stack([sub[c].to_numpy() for c in temp_cols])
    t = sub["timestamp"].to_numpy()
    avg = np.nanmean(temps, axis=1)
    
    return t, avg


def match_by_board(dfPA, dfRB):
    # pulls arrays to numpy
    pa_board = dfPA["board_id"].to_numpy()
    rb_board = dfRB["board_id"].to_numpy()
    t_pa = dfPA["timestamp"].to_numpy()
    t_rb = dfRB["timestamp"].to_numpy()

    # makes a big array with all the indexs -1 just incase a match is not found, what is a "match"?
    idx_out = np.full(len(dfPA), -1, dtype=int)

    # loop over unique boards present in PA
    for b in np.unique(pa_board):
        #the mask takes a board at a time
        pa_mask = (pa_board == b)
        rb_mask = (rb_board == b)

        #skips all the rb1s
        if np.sum(rb_mask) == 0:
            continue  # no matching RB for this board
        
        t_pa_sub = t_pa[pa_mask]
        t_rb_sub = t_rb[rb_mask]
        
        #takes the timestamp arrays of the same boards
        
        idx_sub, dt = nearest_match(t_pa_sub, t_rb_sub)
        
        # map back to full indices
        rb_indices = np.where(rb_mask)[0]
        idx_out[pa_mask] = rb_indices[idx_sub]
    return idx_out


def getTempByBoardAndChannel(dfPA, boardID, channelID):
    col = f"temps{channelID}"

    if col not in dfPA.columns:
        raise ValueError(f"Column {col} not found in dfPA")

    sub = (
        dfPA
        .filter(pl.col("board_id") == boardID)
        .group_by("timestamp")
        .agg(
            pl.col(col).mean().alias(col)
        )
        .sort("timestamp")
    )

    if len(sub) == 0:
        return np.array([]), np.array([])

    t = sub["timestamp"].to_numpy()
    temp = sub[col].to_numpy()

    return t, temp  

def secToHours(sec):
    return sec/3600

In [ ]:

for name, df in {
    "dfPA": dfPA,
    "dfPB": dfPB,
    "dfRB": dfRB,
    "dfSIP": dfSIP,
    "dfRates": dfRates,
    "dfLTB": dfLTB,
    
    
}.items():
    if df is None or len(df) == 0:
        print(f"\n{name}: empty")
        continue

    ##print(f"\n{name}")
    #print("columns:", df.columns)

    time_col = "timestamp" if "timestamp" in df.columns else "total_elapsed"
    
    t = df[time_col].to_numpy()
    ##print("time_col:", time_col)
    #print("first 10:", t[:10])
    #print("last  10:", t[-10:])
    #print("min/max:", np.nanmin(t), np.nanmax(t))

    if "board_id" in df.columns:
        #print("\nfirst few times by board:")
        for b in np.sort(np.unique(df["board_id"].to_numpy()))[:5]:
            sub = df.filter(pl.col("board_id") == b).sort(time_col)
            #print(f"  board {int(b):2d}:", sub[time_col].to_numpy()[:5])

In [ ]:
import inspect

classes = sorted(
    name for name, obj in inspect.getmembers(gon.monitoring)
    if inspect.isclass(obj)
)



print("\n=== gon.monitoring classes ===\n")
for c in classes:
    print("     " + str(c))
    

In [ ]:
boardID = 7
t, temp_avg = getTempAvB(dfPA, boardID)

plt.figure()
plt.plot(t, temp_avg, ".")
plt.title(f"Average PA temperature, board {boardID}")
plt.xlabel("Time since start (s)")
plt.ylabel("Temp (C)")
plt.show()

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------
# helper
# -------------------------------------------------
def secToHours(t):
    return t / 3600.0

def split_into_chunks(arr, chunk_size=8):
    arr = list(arr)
    return [
        arr[i:i + chunk_size]
        for i in range(0, len(arr), chunk_size)
    ]

# -------------------------------------------------
# now using the mapping...
# -------------------------------------------------
pbch_to_paddle = build_pbch_to_paddle_map(paddle_map)

# Sort by absolute paddle number, then side A/B
paddle_keys = sorted(
    paddle_map.keys(),
    key=lambda k: (abs(k), k > 0)
)

paddle_groups = split_into_chunks(paddle_keys, chunk_size=8)

for group_idx, paddle_group in enumerate(paddle_groups):

    plt.figure(figsize=(12, 5))

    group_t0 = None

    for key in paddle_group:

        boardID = paddle_map[key]["rb"]
        channel = paddle_map[key]["ch"]

        t, temp_avg = getTempByBoardAndChannel(
            dfPA,
            boardID,
            channel
        )

        t = np.asarray(t, dtype=float)
        temp_avg = np.asarray(temp_avg, dtype=float)

        m = np.isfinite(t) & np.isfinite(temp_avg)

        if np.sum(m) == 0:
            continue

        t = t[m]
        temp_avg = temp_avg[m]

        if group_t0 is None:
            group_t0 = np.min(t)

        if key > 0:
            sideStr = "B"
        else:
            sideStr = "A"

        label = (
            f"paddle {abs(key)} side {sideStr} "
            f"(RB {boardID}, ch {channel})"
        )

        plt.plot(
            secToHours(t - group_t0),
            temp_avg,
            ".-",
            markersize=3,
            lw=0.8,
            alpha=0.8,
            label=label
        )

    
    plt.title(f"Average PA temperature — paddle group {group_idx}")
    plt.xlabel("Time since group start (hr)")
    plt.ylabel("Temp (C)")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------
# helpers
# -------------------------------------------------
def secToHours(t):
    return t / 3600.0

def split_into_chunks(arr, chunk_size=8):
    arr = list(arr)
    return [
        arr[i:i + chunk_size]
        for i in range(0, len(arr), chunk_size)
    ]

def getSideTemp(key):
    """Finite (t, temp) for a signed paddle key (A = negative, B = positive)."""
    boardID = paddle_map[key]["rb"]
    channel = paddle_map[key]["ch"]

    t, temp = getTempByBoardAndChannel(dfPA, boardID, channel)

    t = np.asarray(t, dtype=float)
    temp = np.asarray(temp, dtype=float)

    m = np.isfinite(t) & np.isfinite(temp)
    return t[m], temp[m]

# -------------------------------------------------
# channels to cut: (rb, ch) pairs to exclude
# -------------------------------------------------
excluded_rb_ch = {
    (32, 3),
    (28, 4),
    (35, 5),
    (46, 7),
    (46, 13),
}

# -------------------------------------------------
# A - B temperature difference for every paddle
# -------------------------------------------------
paddle_numbers = sorted({abs(k) for k in paddle_map.keys()})

paddle_groups = split_into_chunks(paddle_numbers, chunk_size=8)

for group_idx, paddle_group in enumerate(paddle_groups):

    plt.figure(figsize=(12, 5))

    group_t0 = None
    plotted = False

    for pid in paddle_group:

        keyA = -pid   # side A
        keyB = pid    # side B

        if keyA not in paddle_map or keyB not in paddle_map:
            continue

        # cut paddles that use one of the flagged channels on either side
        sides_rb_ch = {
            (paddle_map[keyA]["rb"], paddle_map[keyA]["ch"]),
            (paddle_map[keyB]["rb"], paddle_map[keyB]["ch"]),
        }
        if sides_rb_ch & excluded_rb_ch:
            continue

        tA, tempA = getSideTemp(keyA)
        tB, tempB = getSideTemp(keyB)

        if len(tA) == 0 or len(tB) == 0:
            continue

        # restrict to the overlapping time window of the two sides
        t_lo = max(tA.min(), tB.min())
        t_hi = min(tA.max(), tB.max())

        sel = (tA >= t_lo) & (tA <= t_hi)
        if np.sum(sel) == 0:
            continue

        t_common = tA[sel]

        # interpolate side B onto side A timestamps, then difference
        tempB_i = np.interp(t_common, tB, tempB)
        dT = tempA[sel] - tempB_i

        if group_t0 is None:
            group_t0 = t_common.min()

        rbA, chA = paddle_map[keyA]["rb"], paddle_map[keyA]["ch"]
        rbB, chB = paddle_map[keyB]["rb"], paddle_map[keyB]["ch"]

        label = (
            f"paddle {pid} "
            f"(A: RB {rbA}, ch {chA}  -  B: RB {rbB}, ch {chB})"
        )

        plt.plot(
            secToHours(t_common - group_t0),
            dT,
            ".-",
            markersize=3,
            lw=0.8,
            alpha=0.8,
            label=label
        )
        plotted = True

    if not plotted:
        plt.close()
        continue

    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.title(f"PA temperature difference (side A - side B) — paddle group {group_idx}")
    plt.xlabel("Time since group start (hr)")
    plt.ylabel("Temp difference A - B (C)")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

In [ ]:
def getTempAvB_raw(dfPA, boardID):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]

    sub = (
        dfPA
        .filter(pl.col("board_id") == boardID)
        .sort("timestamp")
    )

    if len(sub) == 0:
        return np.array([]), np.array([])

    temps = np.column_stack([sub[c].to_numpy() for c in temp_cols])
    t = sub["timestamp"].to_numpy()
    avg = np.nanmean(temps, axis=1)

    return t, avg

boardID = 7
t, temp_avg = getTempAvB_raw(dfPA, boardID)

plt.figure(figsize=(8,4))
plt.plot(t - t[0], temp_avg, ".")
plt.title(f"Raw average PA temperature, board {boardID}")
plt.xlabel("Time since start (s)")
plt.ylabel("Temp (C)")
plt.show()
    

# altitude vs rate

In [ ]:
boardID = 7

sub = (
    dfPA
    .filter(pl.col("board_id") == boardID)
    .sort("timestamp")
)

print(
    sub.group_by("timestamp")
       .len()
       .sort("timestamp")
       .head(30)
)
print(dfPA)

In [ ]:


temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
boards = np.unique(dfPA["board_id"].to_numpy())

for b in boards:
    mask = (dfPA["board_id"].to_numpy() == b)

    if np.sum(mask) < 5:
        continue

    t = dfPA["timestamp"].to_numpy()[mask]

    temps = np.column_stack([
        dfPA[c].to_numpy()[mask] for c in temp_cols
    ])

    avg = np.nanmean(temps, axis=1)
    std = np.nanstd(temps, axis=1)

    plt.figure(figsize=(8,5))

    # individual channels
    for i in range(temps.shape[1]):
        plt.scatter(t, temps[:, i], alpha=0.3)

    # average
    plt.plot(t, avg, linewidth=2, label="avg")

    # spread band
    plt.fill_between(t, avg-std, avg+std, alpha=0.2)

    plt.xlabel("Time")
    plt.ylabel("Temp (C)")
    plt.title(f"Board {b} temps (with avg + spread)")
    plt.legend()
    plt.show()

    

In [ ]:
temp_avg = getTempAv(dfPA)

idx = match_by_board(dfPA, dfRB)
valid = idx >= 0

pa_board = dfPA["board_id"].to_numpy()
t_pa = dfPA["timestamp"].to_numpy()
t_rb = dfRB["timestamp"].to_numpy()
rb_rate = dfRB["rate"].to_numpy()



import matplotlib.pyplot as plt
import numpy as np

for b in np.unique(pa_board):

    mask = (pa_board == b) & valid

    if np.sum(mask) < 20:
        continue

    t = t_pa[mask]
    temp = temp_avg[mask]
    rate = rb_rate[idx[mask]]

    fig, axes = plt.subplots(2, 1, figsize=(8,6), sharex=True)

    # -----------------------
    # Temperature vs time
    # -----------------------
    axes[0].plot(t, temp, ".", markersize=3)
    axes[0].set_ylabel("Temp (C)")
    axes[0].set_title(f"Board {int(b)}")

    # -----------------------
    # Rate vs time
    # -----------------------
    axes[1].plot(t, rate, ".", markersize=3)
    axes[1].set_ylabel("Rate")
    axes[1].set_xlabel("Time")

    plt.tight_layout()
    plt.show()
    

# LTB low level checkout 

In [ ]:
# LTB temperature sensors: one figure per temp column, every board overlaid vs time
# Looks good!, 
#behaviouer non trivial so i am not sure if it is fine to reduce to a mean byt we can try so plot is not all that busy


import os

outdir = "low_level_analysis/LTB"
os.makedirs(outdir, exist_ok=True)

os.makedirs("low_level_analysis/LTB/spikes", exist_ok=True)
os.makedirs("low_level_analysis/LTB/overlay", exist_ok=True)
os.makedirs("low_level_analysis/LTB/mean_rms", exist_ok=True)



N_graphs = 3


ltb_temp_cols = ["trenz_temp", "ltb_temp"]



all_boards = np.sort(np.unique(dfLTB["board_id"].to_numpy()))
ltb_ids = np.array([
    RB_to_LTB.get(int(b), int(b))
    for b in all_boards
])


badBoards = [39, 22, 3, 23, 35]


if dfLTB is None or len(dfLTB) == 0:
    print("dfLTB is empty; skip LTB temperature overlay plots.")
else:
    board_col = "board_id"
    time_col = "timestamp"
    
    boards = np.sort(np.unique(dfLTB[board_col].to_numpy()))
    for col in ltb_temp_cols:
        if col not in dfLTB.columns:
            continue
    
        nPlots = int(np.ceil(len(boards) / N_graphs))
    
        for n in range(N_graphs):
            plt.figure(figsize=(12, 4))

            #bad eggs
            # for all
            #for b in boards[int(nPlots*n):int(nPlots*(n+1))]:
            for b in boards[int(nPlots*n):int(nPlots*(n+1))]:
                bLTB = RB_to_LTB.get(int(b), int(b))
    
                sub = (
                    dfLTB
                    .filter(pl.col(board_col) == b)
                    .sort(time_col)
                )
    
                t = sub[time_col].to_numpy()
                y = sub[col].to_numpy()
    
                m = finite_mask(t, y)
    
                if "LTB_TEMP_MIN" in globals() and "LTB_TEMP_MAX" in globals():
                    m &= range_mask(y, LTB_TEMP_MIN, LTB_TEMP_MAX)
    
                plt.plot(
                    t[m],
                    y[m],
                    ".-",
                    markersize=2,
                    lw=0.6,
                    alpha = 0.5,
                    label=f" RB {b}"
                )
    
            plt.xlabel("timestamp")
            plt.ylabel(f"{col} (C)")
            plt.title(f"LTB {col} —  group {n}")
            plt.legend(fontsize=7, ncol=2)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            fname = f"{outdir}/overlay/overlay_{col}_group{n}.png"
            plt.savefig(fname, dpi=150)
            plt.show()
            plt.close()
def summarize_ltb_temp_spikes(
    dfLTB,
    temp_cols=("trenz_temp", "ltb_temp"),
    board_col="board_id",
    time_col="timestamp",
    spike_sigma=5.0,
):
    rows = []
    spike_log = []

    boards = np.sort(np.unique(dfLTB[board_col].to_numpy()))

    for col in temp_cols:
        if col not in dfLTB.columns:
            continue

        for b in boards:
            sub = (
                dfLTB
                .filter(pl.col(board_col) == b)
                .sort(time_col)
            )

            t = sub[time_col].to_numpy()
            y = sub[col].to_numpy()

            m = np.isfinite(t) & np.isfinite(y)
            t = t[m]
            y = y[m]

            if len(y) == 0:
                continue

            mean = np.mean(y)
            rms = np.std(y)

            if rms > 0:
                spike_mask = np.abs(y - mean) > spike_sigma * rms
            
                for ts, val in zip(t[spike_mask], y[spike_mask]):
                    spike_log.append({
                        "column": col,
                        "board_id": int(b),
                        "timestamp": float(ts),
                        "value": float(val),
                        "mean": float(mean),
                        "rms": float(rms),
                    })
            
                n_spikes = np.sum(spike_mask)
            else:
                n_spikes = 0

            rows.append({
                "column": col,
                "board_id": int(b),
                "mean": mean,
                "rms": rms,
                "n_spikes": int(n_spikes),
                "n_points": len(y),
            })

    return rows, spike_log

summary, spike_log = summarize_ltb_temp_spikes(dfLTB, spike_sigma=4.0)

for col in ltb_temp_cols:
    rows = [r for r in summary if r["column"] == col]

    board_map = {r["board_id"]: r for r in rows}

    means = np.array([board_map.get(int(b), {}).get("mean", np.nan) for b in all_boards])
    rms = np.array([board_map.get(int(b), {}).get("rms", np.nan) for b in all_boards])
    spikes = np.array([board_map.get(int(b), {}).get("n_spikes", 0) for b in all_boards])

    # mean ± RMS
    plt.figure(figsize=(10, 4))
    plt.errorbar(ltb_ids, means, yerr=rms, fmt="o", capsize=3)
    plt.xticks(ltb_ids)
    plt.grid(True, alpha=0.3)
    plt.xlabel("LTB board ID")
    plt.ylabel(f"{col} mean ± RMS (C)")
    plt.title(f"LTB {col}: mean and RMS per board")
    plt.tight_layout()
    plt.savefig(f"{outdir}/mean_rms/mean_rms_{col}.png", dpi=150)
    plt.show()
    plt.close()

    # spikes
    plt.figure(figsize=(10, 4))
    plt.bar(ltb_ids, spikes)
    plt.xticks(ltb_ids)
    plt.grid(True, alpha=0.3)
    plt.xlabel("LTB board ID")
    plt.ylabel("number of spikes > 4 RMS")
    plt.title(f"LTB {col}: spike count per board")
    plt.tight_layout()
    plt.savefig(f"{outdir}/spikes/spikes_{col}.png", dpi=150)
    plt.show()
    plt.close()
    

print("\n=== LTB TEMP SPIKES ===")
for s in spike_log:
    print(
        f"board {RB_to_LTB[int(s['board_id'])]} | {s['column']:12s} | "
        f"t = {s['timestamp']:.1f} | value = {s['value']:.4f} C | "
        f"mean = {s['mean']:.4f} C | rms = {s['rms']:.} C"
    )
    
with open(f"{outdir}/spike_log.txt", "w") as f:
    for s in spike_log:
        f.write(
            f"board {RB_to_LTB.get(int(s['board_id']), int(s['board_id']))} | "
            f"{s['column']} | t={s['timestamp']:.1f} | "
            f"value={s['value']:.2f} | mean={s['mean']:.2f} | rms={s['rms']:.2f}\n"
        )

# PB low level check out

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import polars as pl

outdir = "low_level_analysis/PB"

os.makedirs(outdir, exist_ok=True)
os.makedirs(f"{outdir}/spikes", exist_ok=True)
os.makedirs(f"{outdir}/overlay", exist_ok=True)
os.makedirs(f"{outdir}/mean_rms", exist_ok=True)

N_graphs = 3

pb_cols = [
    "p3v6_preamp_v",
    "p3v6_preamp_c",
    "p3v6_preamp_p",
    "n1v6_preamp_v",
    "n1v6_preamp_c",
    "n1v6_preamp_p",
    "p3v4f_ltb_v",
    "p3v4f_ltb_c",
    "p3v4f_ltb_p",
    "p3v4d_ltb_v",
    "p3v4d_ltb_c",
    "p3v4d_ltb_p",
    "p3v6_ltb_v",
    "p3v6_ltb_c",
    "p3v6_ltb_p",
    "n1v6_ltb_v",
    "n1v6_ltb_c",
    "n1v6_ltb_p",
    "pds_temp",
    "pas_temp",
    "nas_temp",
    "shv_temp",
]


def pb_ylabel(col):
    if col.endswith("_v"):
        return f"{col} (V)"
    if col.endswith("_c"):
        return f"{col} (A)"
    if col.endswith("_p"):
        return f"{col} (W)"
    if col.endswith("_temp"):
        return f"{col} (C)"
    return col


# =========================================================
# PB overlays
# =========================================================

if dfPB is None or len(dfPB) == 0:
    print("dfPB is empty; skip PB overlay plots.")
else:
    board_col = "board_id"
    time_col = "timestamp"

    boards = np.sort(np.unique(dfPB[board_col].to_numpy())).astype(int)
    nPlots = int(np.ceil(len(boards) / N_graphs))

    for col in pb_cols:
        if col not in dfPB.columns:
            print(f"Missing {col}; skipping.")
            continue

        for n in range(N_graphs):
            plt.figure(figsize=(12, 4))
            #bad eggs
            for b in boards[int(nPlots*n):int(nPlots*(n+1))]:            # for all
            #for b in boards[int(nPlots*n):int(nPlots*(n+1))]:
                pb_id = RB_to_PB.get(int(b), int(b))

                sub = (
                    dfPB
                    .filter(pl.col(board_col) == b)
                    .sort(time_col)
                )

                t = sub[time_col].to_numpy()
                y = sub[col].to_numpy()

                m = finite_mask(t, y)

                plt.plot(
                    t[m],
                    y[m],
                    ".-",
                    markersize=1.7,
                    lw=0.6,
                    alpha = 0.5,

                    label=f" RB {b}"
                )

            plt.xlabel("timestamp")
            plt.ylabel(pb_ylabel(col))
            plt.title(f"PB {col} — board group {n}")
            plt.legend(loc="best", fontsize=7, ncol=2)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(f"{outdir}/overlay/overlay_{col}_group{n}.png", dpi=150)
            plt.show()
            plt.close()

# =========================================================
# Find PB boards that stop working / have long gaps
# =========================================================

import numpy as np
import polars as pl

GAP_SECONDS = 3600          # 1 hour
FINAL_MISSING_SECONDS = 3600 # board final timestamp is >1 hr before global final timestamp

board_col = "board_id"
time_col = "timestamp"

# ---------------------------------------------------------
# Make sure timestamps are sorted and finite
# ---------------------------------------------------------
dfPB_time = (
    dfPB
    .filter(pl.col(time_col).is_not_null())
    .filter(pl.col(board_col).is_not_null())
    .sort([board_col, time_col])
)

# global final timestamp in the whole PB dataframe
global_t_final = dfPB_time[time_col].max()

print(f"Global final timestamp: {global_t_final}")

bad_gap_log = []
early_stop_log = []

boards = np.sort(
    np.unique(dfPB_time[board_col].to_numpy())
).astype(int)

for b in boards:

    sub = (
        dfPB_time
        .filter(pl.col(board_col) == int(b))
        .sort(time_col)
    )

    t = sub[time_col].to_numpy().astype(float)
    t = t[np.isfinite(t)]

    if len(t) < 2:
        continue

    pb_id = RB_to_PB.get(int(b), int(b))

    # -----------------------------------------------------
    # 1. Find gaps longer than 1 hour
    # -----------------------------------------------------
    dt = np.diff(t)

    gap_idxs = np.where(dt > GAP_SECONDS)[0]

    for idx in gap_idxs:
        bad_gap_log.append({
            "rb_id": int(b),
            "pb_id": int(pb_id),
            "gap_start": t[idx],
            "gap_end": t[idx + 1],
            "gap_seconds": dt[idx],
            "gap_hours": dt[idx] / 3600.0,
        })

    # -----------------------------------------------------
    # 2. Find boards whose last timestamp is early
    # -----------------------------------------------------
    board_t_final = t[-1]
    missing_seconds = global_t_final - board_t_final

    if missing_seconds > FINAL_MISSING_SECONDS:
        early_stop_log.append({
            "rb_id": int(b),
            "pb_id": int(pb_id),
            "board_final_timestamp": board_t_final,
            "global_final_timestamp": global_t_final,
            "missing_seconds": missing_seconds,
            "missing_hours": missing_seconds / 3600.0,
        })


# =========================================================
# Print results
# =========================================================

print("\n=================================================")
print("Boards with timestamp gaps > 1 hour")
print("=================================================")

if len(bad_gap_log) == 0:
    print("No boards had gaps longer than 1 hour.")
else:
    for r in bad_gap_log:
        print(
            f"RB {r['rb_id']:>3} / PB {r['pb_id']:>3}: "
            f"gap from {r['gap_start']:.0f} to {r['gap_end']:.0f} "
            f"= {r['gap_hours']:.2f} hours"
        )


print("\n=================================================")
print("Boards whose final timestamp is > 1 hour before run end")
print("=================================================")

if len(early_stop_log) == 0:
    print("No boards stopped more than 1 hour before the global final timestamp.")
else:
    for r in early_stop_log:
        print(
            f"RB {r['rb_id']:>3} / PB {r['pb_id']:>3}: "
            f"last timestamp = {r['board_final_timestamp']:.0f}, "
            f"global final = {r['global_final_timestamp']:.0f}, "
            f"missing {r['missing_hours']:.2f} hours"
        )
# =========================================================
# PB spike summary
# =========================================================

def summarize_pb_spikes(
    dfPB,
    cols,
    board_col="board_id",
    time_col="timestamp",
    spike_sigma=4.0,
):
    rows = []
    spike_log = []

    boards = np.sort(np.unique(dfPB[board_col].to_numpy())).astype(int)

    for col in cols:
        if col not in dfPB.columns:
            continue

        # skip power columns
        if col.endswith("_p"):
            continue

        
        #bad eggs
        for b in boards:
        # for all
        #for b in boards[int(nPlots*n):int(nPlots*(n+1))]:
            sub = (
                dfPB
                .filter(pl.col(board_col) == b)
                .sort(time_col)
            )

            t = sub[time_col].to_numpy()
            y = sub[col].to_numpy()

            m = np.isfinite(t) & np.isfinite(y)
            t = t[m]
            y = y[m]

            if len(y) == 0:
                continue

            mean = np.mean(y)
            rms = np.std(y)

            if rms > 0:
                # current columns: flag values outside ±10% of mean
                if col.endswith("_c"):
                    spike_mask = np.abs(y - mean) > 0.10 * np.abs(mean)

                # voltage/temp/etc columns: sigma-based spike finder
                else:
                    spike_mask = np.abs(y - mean) > spike_sigma * rms

                for ts, val in zip(t[spike_mask], y[spike_mask]):
                    spike_log.append({
                        "column": col,
                        "board_id": int(b),
                        "timestamp": float(ts),
                        "value": float(val),
                        "mean": float(mean),
                        "rms": float(rms),
                    })

                n_spikes = np.sum(spike_mask)
            else:
                n_spikes = 0

            rows.append({
                "column": col,
                "board_id": int(b),
                "mean": float(mean),
                "rms": float(rms),
                "n_spikes": int(n_spikes),
                "n_points": int(len(y)),
            })

    return rows, spike_log


summary_pb, spike_log_pb = summarize_pb_spikes(
    dfPB,
    pb_cols,
    spike_sigma=4.0,
)


# =========================================================
# PB mean/RMS and spike count plots
# =========================================================

all_pb_boards = np.sort(np.unique(dfPB["board_id"].to_numpy())).astype(int)

pb_ids = np.array([
    RB_to_PB.get(int(b), int(b))
    for b in all_pb_boards
])

for col in pb_cols:
    if col not in dfPB.columns:
        continue

    # skip power columns because they were skipped in summary
    if col.endswith("_p"):
        continue

    rows = sorted(
        [r for r in summary_pb if r["column"] == col],
        key=lambda r: int(r["board_id"])
    )

    board_map = {
        int(r["board_id"]): r
        for r in rows
    }

    means = np.array([
        board_map.get(int(b), {}).get("mean", np.nan)
        for b in all_pb_boards
    ])

    rms = np.array([
        board_map.get(int(b), {}).get("rms", np.nan)
        for b in all_pb_boards
    ])

    spikes = np.array([
        board_map.get(int(b), {}).get("n_spikes", 0)
        for b in all_pb_boards
    ])

    # mean ± RMS
    plt.figure(figsize=(10, 4))
    plt.errorbar(pb_ids, means, yerr=rms, fmt="o", capsize=3)
    plt.xticks(pb_ids)
    plt.grid(True, which="both", axis="both", alpha=0.3)
    plt.xlabel("PB board ID")
    plt.ylabel(f"{pb_ylabel(col)} mean ± RMS")
    plt.title(f"PB {col}: mean and RMS per board")
    plt.tight_layout()
    plt.savefig(f"{outdir}/mean_rms/mean_rms_{col}.png", dpi=150)
    plt.show()
    plt.close()

    print("pb_ids")
    print(pb_ids)
    print("spikes")
    print(spikes)
    print("mean")
    print(means)

    # spike counts
    plt.figure(figsize=(10, 4))
    plt.bar(pb_ids, spikes)
    plt.xticks(pb_ids)
    plt.grid(True, which="both", axis="both", alpha=0.3)
    plt.xlabel("PB board ID")

    if col.endswith("_c"):
        plt.ylabel("number of spikes > 10% mean")
    else:
        plt.ylabel("number of spikes > 4 RMS")

    plt.title(f"PB {col}: spike count per board")
    plt.tight_layout()
    plt.savefig(f"{outdir}/spikes/spikes_{col}.png", dpi=150)
    plt.show()
    plt.close()


# =========================================================
# Print and save spike log
# =========================================================

print("\n=== PB SPIKES ===")

with open(f"{outdir}/spike_log.txt", "w") as f:
    for s in spike_log_pb:
        pb_id = RB_to_PB.get(int(s["board_id"]), int(s["board_id"]))

        line = (
            f"board {pb_id} | {s['column']:16s} | "
            f"t = {s['timestamp']:.1f} | "
            f"value = {s['value']:.5f} | "
            f"mean = {s['mean']:.5f} | "
            f"rms = {s['rms']:.5f}\n"
        )

        print(line, end="")
        f.write(line)

# RB low level checkout

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import polars as pl


# -------------------------------------------------
# setup
# -------------------------------------------------
outdir = "low_level_analysis/RB"

os.makedirs(outdir, exist_ok=True)
os.makedirs(f"{outdir}/spikes", exist_ok=True)
os.makedirs(f"{outdir}/overlay", exist_ok=True)
os.makedirs(f"{outdir}/mean_rms", exist_ok=True)

N_graphs = 5


def split_into_groups(arr, n_groups=5):
    arr = np.sort(np.asarray(arr).astype(int))
    return [g for g in np.array_split(arr, n_groups) if len(g) > 0]


def safe_filename(s):
    return str(s).replace("/", "_").replace(" ", "_").replace(":", "_")


# -------------------------------------------------
# RB columns + labels
# -------------------------------------------------
rb_cols = [
    "tmp_drs","tmp_clk","tmp_adc","tmp_zynq","tmp_lis3mdltr","tmp_bm280",

    "drs_dvdd_voltage","drs_dvdd_current","drs_dvdd_power",
    "p3v3_voltage","p3v3_current","p3v3_power",
    "zynq_voltage","zynq_current","zynq_power",
    "p3v5_voltage","p3v5_current","p3v5_power",

    "adc_dvdd_voltage","adc_dvdd_current","adc_dvdd_power",
    "adc_avdd_voltage","adc_avdd_current","adc_avdd_power",

    "drs_avdd_voltage","drs_avdd_current","drs_avdd_power",
    "n1v5_voltage","n1v5_current","n1v5_power",
]


def rb_ylabel(col):
    if col.startswith("tmp_"):
        return f"{col} (C)"
    if col.endswith("_voltage"):
        return f"{col} (V)"
    if col.endswith("_current"):
        return f"{col} (A)"
    if col.endswith("_power"):
        return f"{col} (W)"
    return col

# -------------------------------------------------
# Overlay plots
# -------------------------------------------------
if dfRB is None or len(dfRB) == 0:
    print("dfRB is empty; skip RB overlay plots.")
else:
    board_col = "board_id"
    time_col = "timestamp"

    all_rb_boards = np.sort(
        np.unique(dfRB[board_col].to_numpy())
    ).astype(int)

    rb_board_groups = split_into_groups(all_rb_boards, N_graphs)

    for col in rb_cols:
        if col not in dfRB.columns:
            print(f"Missing {col}; skipping.")
            continue

        for group_idx, boards in enumerate(rb_board_groups):

            plt.figure(figsize=(12, 4))

            for b in boards:

                sub = (
                    dfRB
                    .filter(pl.col(board_col) == int(b))
                    .sort(time_col)
                )

                if len(sub) == 0:
                    continue

                t = sub[time_col].to_numpy()
                y = sub[col].to_numpy().astype(float)

                m = finite_mask(t, y)

                # remove absurd values
                m &= np.isfinite(y)
                m &= np.abs(y) < 1e4

                t = t[m]
                y = y[m]

                if len(y) == 0:
                    continue

                plt.plot(
                    t,
                    y,
                    ".-",
                    markersize=2,
                    lw=0.6,
                    alpha=0.5,
                    label=f"RB {b}"
                )

            plt.xlabel("timestamp")
            plt.ylabel(rb_ylabel(col))
            plt.title(f"RB {col} — boards group {group_idx}")
            plt.legend(fontsize=7, ncol=2)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()

            plt.savefig(
                f"{outdir}/overlay/overlay_{safe_filename(col)}_group{group_idx}.png",
                dpi=150
            )

            plt.show()
            plt.close()
# -------------------------------------------------
# Spike summary
# -------------------------------------------------
def summarize_rb_spikes(
    dfRB,
    cols,
    board_col="board_id",
    time_col="timestamp",
    spike_sigma=4.0,
):
    rows = []
    spike_log = []

    boards = np.sort(
        np.unique(dfRB[board_col].to_numpy())
    ).astype(int)

    for col in cols:
        if col not in dfRB.columns:
            continue

        # skip power columns
        if col.endswith("_power"):
            continue

        for b in boards:
            if b == 19:
                print("skipping 19")
                continue
            sub = (
                dfRB
                .filter(pl.col(board_col) == int(b))
                .sort(time_col)
            )

            t = sub[time_col].to_numpy()
            y = sub[col].to_numpy()

            m = np.isfinite(t) & np.isfinite(y)
            t = t[m]
            y = y[m]

            if len(y) == 0:
                continue

            mean = np.mean(y)
            rms = np.std(y)

            if rms > 0:
                # current columns: flag values outside ±10% of mean
                if col.endswith("_current"):
                    spike_mask = np.abs(y - mean) > 0.10 * np.abs(mean)

                # voltage/temp/etc columns: sigma-based spike finder
                else:
                    spike_mask = np.abs(y - mean) > spike_sigma * rms

                for ts, val in zip(t[spike_mask], y[spike_mask]):
                    spike_log.append({
                        "column": col,
                        "board_id": int(b),
                        "timestamp": float(ts),
                        "value": float(val),
                        "mean": float(mean),
                        "rms": float(rms),
                    })

                n_spikes = np.sum(spike_mask)
            else:
                n_spikes = 0

            rows.append({
                "column": col,
                "board_id": int(b),
                "mean": float(mean),
                "rms": float(rms),
                "n_spikes": int(n_spikes),
                "n_points": int(len(y)),
            })

    return rows, spike_log


summary_rb, spike_log_rb = summarize_rb_spikes(
    dfRB,
    rb_cols,
    spike_sigma=4.0,
)


# -------------------------------------------------
# Mean/RMS + spike plots
# -------------------------------------------------
all_rb_boards = np.sort(
    np.unique(dfRB["board_id"].to_numpy())
).astype(int)

for col in rb_cols:
    if col not in dfRB.columns:
        continue

    # skip power columns because they were skipped in summary
    if col.endswith("_power"):
        continue

    rows = sorted(
        [r for r in summary_rb if r["column"] == col],
        key=lambda r: int(r["board_id"])
    )

    board_map = {
        int(r["board_id"]): r
        for r in rows
    }

    means = np.array([
        board_map.get(int(b), {}).get("mean", np.nan)
        for b in all_rb_boards
    ])

    rms = np.array([
        board_map.get(int(b), {}).get("rms", np.nan)
        for b in all_rb_boards
    ])

    spikes = np.array([
        board_map.get(int(b), {}).get("n_spikes", 0)
        for b in all_rb_boards
    ])
    
    # mean ± RMS
    plt.figure(figsize=(12, 4))
    plt.errorbar(all_rb_boards, means, yerr=rms, fmt="o", capsize=3)
    plt.xticks(all_rb_boards)
    plt.grid(True, which="both", axis="both", alpha=0.3)
    plt.xlabel("RB board ID")
    plt.ylabel(f"{rb_ylabel(col)} mean ± RMS")
    plt.title(f"RB {col}: mean and RMS per board")
    plt.tight_layout()

    plt.savefig(
        f"{outdir}/mean_rms/mean_rms_{safe_filename(col)}.png",
        dpi=150
    )
    plt.show()
    plt.close()

    print("rb_ids")
    print(all_rb_boards)
    print("spikes")
    print(spikes)
    print("mean")
    print(means)

    # spike count
    plt.figure(figsize=(12, 4))
    plt.bar(all_rb_boards, spikes)
    plt.xticks(all_rb_boards)
    plt.grid(True, which="both", axis="both", alpha=0.3)
    plt.xlabel("RB board ID")

    if col.endswith("_current"):
        plt.ylabel("number of spikes > 10% mean")
    else:
        plt.ylabel("number of spikes > 4 RMS")

    plt.title(f"RB {col}: spike count per board")
    plt.tight_layout()

    plt.savefig(
        f"{outdir}/spikes/spikes_{safe_filename(col)}.png",
        dpi=150
    )
    plt.show()
    plt.close()


# -------------------------------------------------
# Print + save spike log
# -------------------------------------------------
print("\n=== RB SPIKES ===")

with open(f"{outdir}/spike_log.txt", "w") as f:
    for s in spike_log_rb:
        line = (
            f"board {s['board_id']} | {s['column']:20s} | "
            f"t = {s['timestamp']:.1f} | "
            f"value = {s['value']:.5f} | "
            f"mean = {s['mean']:.5f} | "
            f"rms = {s['rms']:.5f}\n"
        )

        print(line, end="")
        f.write(line)

# PA low level checkout

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import polars as pl

# -------------------------------------------------
# setup
# -------------------------------------------------
outdir = "low_level_analysis/PA"

os.makedirs(outdir, exist_ok=True)
os.makedirs(f"{outdir}/spikes", exist_ok=True)
os.makedirs(f"{outdir}/overlay", exist_ok=True)
os.makedirs(f"{outdir}/mean_rms", exist_ok=True)

N_graphs = 3

# If these are not already defined elsewhere, set defaults
try:
    badBoards
except NameError:
    badBoards = None

try:
    mean_z_cut
except NameError:
    mean_z_cut = 3.0

try:
    mean_outlier_log_pa
except NameError:
    mean_outlier_log_pa = []


def split_into_groups(arr, n_groups=3):
    arr = np.sort(np.asarray(arr).astype(int))
    return [g for g in np.array_split(arr, n_groups) if len(g) > 0]


def safe_filename(s):
    return (
        str(s)
        .replace("/", "_")
        .replace(" ", "_")
        .replace(":", "_")
    )


def finite_mask(*arrays):
    """
    Return a mask that is True where all arrays are finite.
    """
    mask = np.ones(len(arrays[0]), dtype=bool)
    for a in arrays:
        mask &= np.isfinite(a)
    return mask


# -------------------------------------------------
# PA columns + labels
# -------------------------------------------------
pa_cols = (
    [f"temps{i}" for i in range(1, 17)] +
    [f"biases{i}" for i in range(1, 17)]
)


def pa_ylabel(col):
    if col.startswith("temps"):
        return f"{col} (C)"
    if col.startswith("biases"):
        return f"{col} (V)"
    return col


# =================================================
# Main PA analysis
# =================================================
if dfPA is None or len(dfPA) == 0:
    print("dfPA is empty; skip PA plots.")
else:
    board_col = "board_id"
    time_col = "timestamp"

    all_pa_boards = np.sort(
        np.unique(dfPA[board_col].to_numpy())
    ).astype(int)

    pa_board_groups = split_into_groups(all_pa_boards, N_graphs)

    # -------------------------------------------------
    # Overlay plots
    # -------------------------------------------------
    for col in pa_cols:
        if col not in dfPA.columns:
            print(f"Missing {col}; skipping.")
            continue

        for group_idx, boards in enumerate(pa_board_groups):

            plt.figure(figsize=(12, 4))

            for b in boards:

                sub = (
                    dfPA
                    .filter(pl.col(board_col) == int(b))
                    .sort(time_col)
                )

                if len(sub) == 0:
                    continue

                t = sub[time_col].to_numpy()
                y = sub[col].to_numpy()

                m = finite_mask(t, y)

                plt.plot(
                    t[m],
                    y[m],
                    ".-",
                    markersize=2,
                    lw=0.6,
                    alpha=0.5,
                    label=f"board {int(b)}"
                )

            plt.xlabel("timestamp")
            plt.ylabel(pa_ylabel(col))
            plt.title(f"PA {col} — boards group {group_idx}")
            plt.legend(fontsize=7, ncol=2)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()

            plt.savefig(
                f"{outdir}/overlay/overlay_{safe_filename(col)}_group{group_idx}.png",
                dpi=150
            )

            plt.show()
            plt.close()


    # -------------------------------------------------
    # Spike summary
    # -------------------------------------------------
    def summarize_pa_spikes(
        dfPA,
        cols,
        board_col="board_id",
        time_col="timestamp",
        spike_sigma=4.0,
        boards_to_use=None,
    ):
        rows = []
        spike_log = []

        all_boards = np.sort(
            np.unique(dfPA[board_col].to_numpy())
        ).astype(int)

        if boards_to_use is None:
            boards = all_boards
        else:
            boards = np.asarray(boards_to_use).astype(int)

        for col in cols:
            if col not in dfPA.columns:
                continue

            for b in boards:

                sub = (
                    dfPA
                    .filter(pl.col(board_col) == int(b))
                    .sort(time_col)
                )

                if len(sub) == 0:
                    continue

                t = sub[time_col].to_numpy()
                y = sub[col].to_numpy()

                m = finite_mask(t, y)
                t = t[m]
                y = y[m]

                if len(y) == 0:
                    continue

                mean = np.mean(y)
                rms = np.std(y)

                if rms > 0 and np.isfinite(rms):
                    spike_mask = np.abs(y - mean) > spike_sigma * rms

                    for ts, val in zip(t[spike_mask], y[spike_mask]):
                        spike_log.append({
                            "column": col,
                            "board_id": int(b),
                            "timestamp": float(ts),
                            "value": float(val),
                            "mean": float(mean),
                            "rms": float(rms),
                        })

                    n_spikes = int(np.sum(spike_mask))
                else:
                    n_spikes = 0

                rows.append({
                    "column": col,
                    "board_id": int(b),
                    "mean": float(mean),
                    "rms": float(rms),
                    "n_spikes": int(n_spikes),
                    "n_points": int(len(y)),
                })

        return rows, spike_log


    # Use badBoards only if you specifically defined it.
    # Otherwise summarize all PA boards.
    summary_pa, spike_log_pa = summarize_pa_spikes(
        dfPA,
        pa_cols,
        spike_sigma=4.0,
        boards_to_use=None,
    )


    # -------------------------------------------------
    # Mean/RMS + spike plots
    # -------------------------------------------------
    for col in pa_cols:
        if col not in dfPA.columns:
            continue

        rows = sorted(
            [r for r in summary_pa if r["column"] == col],
            key=lambda r: int(r["board_id"])
        )

        board_map = {
            int(r["board_id"]): r
            for r in rows
        }

        means = np.array([
            board_map.get(int(b), {}).get("mean", np.nan)
            for b in all_pa_boards
        ])

        rms = np.array([
            board_map.get(int(b), {}).get("rms", np.nan)
            for b in all_pa_boards
        ])

        spikes = np.array([
            board_map.get(int(b), {}).get("n_spikes", 0)
            for b in all_pa_boards
        ])

        # -------------------------------------------------
        # Find boards whose mean is weird compared to others
        # -------------------------------------------------
        good = np.isfinite(means)

        if np.sum(good) >= 3:
            mean_of_means = np.nanmean(means[good])
            std_of_means = np.nanstd(means[good])

            if std_of_means > 0 and np.isfinite(std_of_means):
                z = (means - mean_of_means) / std_of_means
                outlier_mask = good & (np.abs(z) > mean_z_cut)

                for b, mean_val, z_val in zip(
                    all_pa_boards[outlier_mask],
                    means[outlier_mask],
                    z[outlier_mask],
                ):
                    mean_outlier_log_pa.append({
                        "board_id": int(b),
                        "column": col,
                        "mean": float(mean_val),
                        "mean_of_means": float(mean_of_means),
                        "std_of_means": float(std_of_means),
                        "zscore": float(z_val),
                    })

        # -------------------------------------------------
        # Mean ± RMS plot
        # -------------------------------------------------
        plt.figure(figsize=(12, 4))

        plt.errorbar(
            all_pa_boards,
            means,
            yerr=rms,
            fmt="o",
            capsize=3
        )

        plt.xticks(all_pa_boards)
        plt.grid(True, which="both", axis="both", alpha=0.3)
        plt.xlabel("RB control board ID")
        plt.ylabel(f"{pa_ylabel(col)} mean ± RMS")
        plt.title(f"PA {col}: mean and RMS per board")
        plt.tight_layout()

        plt.savefig(
            f"{outdir}/mean_rms/mean_rms_{safe_filename(col)}.png",
            dpi=150
        )

        plt.show()
        plt.close()

        # -------------------------------------------------
        # Spike count plot
        # -------------------------------------------------
        plt.figure(figsize=(12, 4))

        plt.bar(all_pa_boards, spikes)

        plt.xticks(all_pa_boards)
        plt.grid(True, which="both", axis="both", alpha=0.3)
        plt.xlabel("RB control board ID")
        plt.ylabel("number of spikes > 4 RMS")
        plt.title(f"PA {col}: spike count per board")
        plt.tight_layout()

        plt.savefig(
            f"{outdir}/spikes/spikes_{safe_filename(col)}.png",
            dpi=150
        )

        plt.show()
        plt.close()


    # -------------------------------------------------
    # Print + save spike log
    # -------------------------------------------------
    print("\n=== PA SPIKES ===")

    with open(f"{outdir}/spike_log.txt", "w") as f:
        for s in spike_log_pa:
            line = (
                f"control RB board {s['board_id']:2d} | "
                f"{s['column']:10s} | "
                f"t = {s['timestamp']:.1f} | "
                f"value = {s['value']:.3f} | "
                f"mean = {s['mean']:.3f} | "
                f"rms = {s['rms']:.3f}\n"
            )

            print(line, end="")
            f.write(line)


    # -------------------------------------------------
    # Print + save mean outlier log
    # -------------------------------------------------
    print("\n=== PA MEAN OUTLIERS ===")

    with open(f"{outdir}/mean_outlier_log.txt", "w") as f:
        for s in mean_outlier_log_pa:
            line = (
                f"control RB board {s['board_id']:2d} | "
                f"{s['column']:10s} | "
                f"mean = {s['mean']:.3f} | "
                f"mean of boards = {s['mean_of_means']:.3f} | "
                f"std of boards = {s['std_of_means']:.3f} | "
                f"z = {s['zscore']:.2f}\n"
            )

            print(line, end="")
            f.write(line)


In [ ]:
# =========================================================
# Print PA readout voltage means in paddle / side order
# Format:
# paddle_id    side    voltage_mean
# =========================================================

rows_out = []

for key in sorted(paddle_map.keys(), key=lambda k: (abs(k), k > 0)):
    paddle_id = abs(key)

    if key < 0:
        side = "A"
    else:
        side = "B"

    boardID = int(paddle_map[key]["rb"])
    channel = int(paddle_map[key]["ch"])

    col = f"biases{channel}"

    # find the matching mean from summary_pa
    matches = [
        r for r in summary_pa
        if int(r["board_id"]) == boardID and r["column"] == col
    ]

    if len(matches) == 0:
        voltage_mean = np.nan
    else:
        voltage_mean = matches[0]["rms"]

    rows_out.append((paddle_id, side, voltage_mean, boardID, channel))


# ---------------------------------------------------------
# print version for spreadsheet: paddle, side, voltage
# ---------------------------------------------------------
for paddle_id, side, voltage_mean, boardID, channel in rows_out:
    print(f"{voltage_mean:.3f}")